In [1]:
from cartridges.data.qasper.generate_qasper_abstracts import generate_qasper_init_text

saved_path = generate_qasper_init_text( p=512, tokenizer_name="meta-llama/llama-3.2-3B-Instruct", path="/home/vo43/cartridges/examples/qasper/qasper_init_512.txt")

/home/vo43/.conda/envs/cartridges/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading QASPER (train split)…
  [1908.06606] 82/6553 tokens kept
  [1704.05572] 82/4367 tokens kept
  [1905.08949] 82/7516 tokens kept
  [1808.09920] 82/7705 tokens kept
  [1603.01417] 82/7868 tokens kept
  [1808.03986] 82/8750 tokens kept
  [1907.08501] 82/4125 tokens kept
  [1603.07044] 82/5020 tokens kept
  [1903.00172] 82/6995 tokens kept
  [1912.01046] 82/5056 tokens kept
  [1909.00542] 82/3863 tokens kept
  [1811.08048] 82/6756 tokens kept
  [2004.02393] 82/4933 tokens kept
  [1703.06492] 82/6967 tokens kept
  [1607.06275] 82/5788 tokens kept
  [1703.04617] 82/7230 tokens kept


In [1]:
from huggingface_hub import HfApi, login

api = HfApi()

In [2]:
from huggingface_hub import HfApi, create_repo

api = HfApi()
repo_id = "qtris123/qwen_0_longhealth-p1-10_8192_no-cartridge" 


# 1. Point directly to your specific file
local_file_path = "/scratch/scholar/vo43/qwen_0_longhealth.parquet"
# 2. Create the empty dataset repository (if it doesn't exist yet)
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

# 3. Upload just the single file
api.upload_file(
    path_or_fileobj=local_file_path,
    path_in_repo="dataset.parquet", # This is what the file will be named on Hugging Face
    repo_id=repo_id,
    repo_type="dataset",
)

print("Single Parquet file upload complete!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Single Parquet file upload complete!


In [6]:
from huggingface_hub import HfApi, create_repo

api = HfApi()
repo_id = "qtris123/llama_longhealth-p11-20_8192_10-epochs_with-cartridge_p1-10_8192_2048"
# llama_qasper-QA-task_8192_2048_no-cartridge_10-epochs
folder_path = "/home/vo43/cartridges/outputs/2026-03-31-12-54-53-arxiv_train/a7f4648d-7351-4fdb-809f-4383f0a73798" # Update this to your actual folder path

# 1. Create the empty repository on Hugging Face first
# exist_ok=True ensures it won't crash if you run this script twice
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 2. Now run your upload command
api.upload_folder(
    folder_path=folder_path,
    repo_id=repo_id,
    repo_type="model",
)
print("Upload complete!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete!


In [63]:
from huggingface_hub import hf_hub_download

# This downloads just the checkpoint file and returns the local path to it
checkpoint_path = hf_hub_download(
    repo_id="qtris123/llama__qasper-QA-task_8192_1024_no-cartridge",
    filename="cache-step53.pt"
)

import torch
# Now you can load it into your model logic
state_dict = torch.load(checkpoint_path)
print(f"Loaded cartridge from {checkpoint_path}")

cache-step53.pt:   0%|          | 0.00/117M [00:00<?, ?B/s]

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.nn.modules.container.ParameterList was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ParameterList])` or the `torch.serialization.safe_globals([ParameterList])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types acce

In [ ]:
state_dict = torch.load(checkpoint_path, weights_only=False, map_location=torch.device('cpu'))
print(f"Loaded cartridge from {checkpoint_path}")
/scratch/scholar/vo43/huggingface/hub/models--qtris123--llama__qasper-QA-task_8192_1024_no-cartridge/snapshots/b9ac9b9274643f8bad8d78865b42011b3d8b3335/cache-step53.pt

Loaded cartridge from /scratch/scholar/vo43/huggingface/hub/models--qtris123--llama__qasper-QA-task_8192_1024_no-cartridge/snapshots/b9ac9b9274643f8bad8d78865b42011b3d8b3335/cache-step53.pt


In [29]:
import requests

# Make a request with a cartridge from HuggingFace
response = requests.post("http://scholar-j000:10210/v1/cartridge/chat/completions", json={
    "model": "default",
    "messages": [{"role": "user", "content": "Help me understand this."}],
    "max_tokens": 50,
    "cartridges": [{
        "id": "hazyresearch/cartridge-wauoq23f",
        "source": "huggingface",
        "force_redownload": False
    }]
})

print(response.json())

{'detail': 'Not Found'}


In [39]:
import requests

# Make a request with a cartridge from HuggingFace
response = requests.post("http://scholar-j000:10210/custom/synchronous-batch-completions", json={
    "model": "default",
    "messages": [{"role": "user", "content": "Help me understand this."}],
})

In [ ]:
request = {
    "messages": chat,
    "model": self.config.model_name,
    "max_completion_tokens": max_completion_tokens,
    "temperature": temperature,
    "apply_chat_template_overrides": thinking_overrides,
    "logprobs_in_fingerprint": True,
}

In [ ]:
import requests

# Make a request with a cartridge from HuggingFace
response = requests.post("http://scholar-j000:10210/v1/cartridge/chat/completions", json={
    "model": "default",
    "messages": [{"role": "user", "content": "Help me understand this."}],
    "max_tokens": 50,
    "cartridges": [{
        "id": "qtris123/qasper-QA-task_8192_1024_no-cartridge",
        "source": "huggingface",
        "force_redownload": False
    }]
})

In [40]:
response.json

<bound method Response.json of <Response [422]>>

In [1]:
from generate_qasper_abstracts import generate_qasper_init_text

init_text = generate_qasper_init_text(p=2048, tokenizer_name="Qwen/Qwen3-4B") # default of paper choices is Part 1


Loading QASPER (train split)…
  [1908.06606] 391/6691 tokens kept
  [1704.05572] 391/4420 tokens kept
  [1905.08949] 391/7654 tokens kept
  [1808.09920] 391/7822 tokens kept
  [1603.01417] 391/8016 tokens kept
  [1808.03986] 391/9017 tokens kept


In [ ]:
init_text

In [2]:
import html
from IPython.display import HTML

HTML(f"<pre>{html.escape(init_text)}</pre>")

In [2]:

from typing import Dict, List, Optional
import json
from pydantic import BaseModel
import requests


class LongHealthAnswerLocation(BaseModel):
    start: List[float]
    end: List[float]


# dict_keys(['No', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct', 'answer_location'])
class LongHealthQuestion(BaseModel):
    question_id: str
    question: str
    correct: str

    answer_a: str
    answer_b: str
    answer_c: str
    answer_d: str
    answer_e: str

    # text_id -> answer_location in that text
    answer_location: Optional[Dict[str, LongHealthAnswerLocation]]
class LongHealthPatient(BaseModel):
    patient_id: str
    texts: Dict[str, str]
    name: str 
    birthday: str
    diagnosis: str
    questions: List[LongHealthQuestion]


    
DATASET_PATH = "https://raw.githubusercontent.com/kbressem/LongHealth/refs/heads/main/data/benchmark_v5.json"

def load_longhealth_dataset(patient_ids: Optional[List[str]] = None) -> List[LongHealthPatient]:
    response = requests.get(DATASET_PATH)
    response.raise_for_status()  # Raise an exception for HTTP errors
    data = json.loads(response.text)
    # rename no to idx
    for patient_id, row in data.items():
        for question in row["questions"]:
            question["question_id"] = patient_id + "_" + str(question["No"])

    patients = [
        LongHealthPatient(
            patient_id=patient_id,
            **row
        )
        for patient_id, row in data.items()
        if patient_ids is None or patient_id in patient_ids
    ]
    print("Num patients:", len(patients))
    return patients

In [3]:
patient_ids = ["patient_11", "patient_12", "patient_13", "patient_14", "patient_15", "patient_16", "patient_17", "patient_18", "patient_19", "patient_20"]
dataset = load_longhealth_dataset(patient_ids)

Num patients: 10


In [46]:
len(dataset[2].texts['text_5'])

11379